# Game of Thrones Death Prediction — Scene Death Modeling

This notebook trains and evaluates five classifiers on the task of predicting whether a Game of Thrones scene contains a character death.

**Target column:** `death_in_scene` (1 = at least one character dies, 0 = no death)

**Data source:** `reports/scenes.parquet` — the cleaned, one-hot encoded scenes DataFrame produced by `src.data`.

Once the best scene-level model is selected, its predictions are aggregated at the episode level and compared against the ground-truth deaths-per-episode data.

Models trained:
1. Decision Tree
2. Random Forest
3. Support Vector Classifier (SVC)
4. Naive Bayes
5. Logistic Regression

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

import sys, pathlib
sys.path.insert(0, str(pathlib.Path("..").resolve()))
from src.models import split, evaluate

## Load Data

In [ ]:
scenes   = pd.read_parquet("../reports/scenes.parquet")
episodes = pd.read_parquet("../reports/episodes.parquet")

print(f"Scenes shape:   {scenes.shape}")
print(f"Episodes shape: {episodes.shape}")
scenes.head()

## Class Balance and Baseline

The scene-death classes are heavily imbalanced — most scenes do not contain a character death. The majority-class baseline is above 0.95, which means the bar for a useful model is high.

In [ ]:
class_counts = scenes['death_in_scene'].value_counts()
scenes_baseline = class_counts.max() / len(scenes)

class_counts.plot.bar()
plt.ylabel('Number of Scenes')
plt.xlabel('death_in_scene')
plt.title('Class Balance — death_in_scene')
plt.tight_layout()
plt.show()

print(f"Majority-class baseline accuracy: {scenes_baseline:.4f}")

## Prepare Features

Drop non-feature columns (character names list, episode title, episode ID) and select only numeric/boolean columns, then split using the `split` helper from `src.models`.

In [ ]:
TARGET = "death_in_scene"

drop_cols = ["character_names", "episodeTitle", "episode_id"]
df = scenes.drop(columns=[c for c in drop_cols if c in scenes.columns])

X_train, X_val, X_test, y_train, y_val, y_test = split(df, TARGET, random_state=42)

print(f"Train: {X_train.shape[0]}  Val: {X_val.shape[0]}  Test: {X_test.shape[0]}")
print(f"Features: {X_train.shape[1]}")

## 1. Decision Tree

I first attempted to predict scene deaths using a decision tree. Because the classes are imbalanced, I used `class_weight='balanced'` which noticeably improved results. Without balancing, accuracy appeared high but was driven almost entirely by predicting the majority class.

Best hyperparameters: `criterion='gini'`, `splitter='random'`, `class_weight='balanced'`.

In [ ]:
dt = DecisionTreeClassifier(criterion='gini', splitter='random', class_weight='balanced', random_state=42)
dt.fit(X_train, y_train)

val_acc  = metrics.accuracy_score(y_val,  dt.predict(X_val))
test_acc = metrics.accuracy_score(y_test, dt.predict(X_test))

print(f"Decision Tree — val accuracy: {val_acc:.4f}  |  test accuracy: {test_acc:.4f}")
print(f"Baseline: {scenes_baseline:.4f}  (decision tree beats baseline: {test_acc > scenes_baseline})")

In [ ]:
print(metrics.classification_report(y_test, dt.predict(X_test), zero_division=0))

Although these accuracy values seem high in absolute terms, when compared to the baseline classifier, the decision tree actually performs worse — it sacrifices majority-class accuracy in exchange for better recall on the minority class.

## 2. Random Forest

The random forest with balanced class weights achieved results near 1.0 on the validation set (0.9983). On the test set it scored ~0.965, which is slightly better than the 0.952 baseline.

Since the baseline was already above 95%, even a 0.01 improvement is meaningful.

In [ ]:
X_train_rf, X_val_rf, X_test_rf, y_train_rf, y_val_rf, y_test_rf = split(
    df, TARGET, random_state=42
)

forest = RandomForestClassifier(class_weight='balanced', random_state=42)
forest.fit(X_train_rf, y_train_rf)

val_acc  = metrics.accuracy_score(y_val_rf,  forest.predict(X_val_rf))
test_acc = metrics.accuracy_score(y_test_rf, forest.predict(X_test_rf))

print(f"Random Forest — val accuracy: {val_acc:.4f}  |  test accuracy: {test_acc:.4f}")
print(f"Baseline: {scenes_baseline:.4f}  (random forest beats baseline: {test_acc > scenes_baseline})")

In [ ]:
print(metrics.classification_report(y_test_rf, forest.predict(X_test_rf), zero_division=0))

### Feature importances

In [ ]:
importances = pd.Series(forest.feature_importances_, index=X_train_rf.columns)
top_features = importances.sort_values(ascending=False).head(15)

top_features.plot.barh()
plt.xlabel('Feature Importance')
plt.title('Top 15 Features — Random Forest (Scene Death Prediction)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Support Vector Classifier

I looked at the correlation matrix as a sanity check before training an SVC. I didn't notice any suspiciously high correlations between features.

I manually tested several kernel/hyperparameter combinations (a grid search was too slow). The sigmoid kernel with `C=1`, `coef0=1` gave the best validation accuracy (~0.945) and a test accuracy of ~0.952, slightly above the baseline.

In [ ]:
# Correlation matrix — eyeball check
corr = df.drop(columns=[TARGET]).corr()
print("Top feature correlations:")
corr.abs().unstack().sort_values(ascending=False).drop_duplicates().head(20)

In [ ]:
X_train_svc, X_val_svc, X_test_svc, y_train_svc, y_val_svc, y_test_svc = split(
    df, TARGET, random_state=42
)

svc = Pipeline([
    ("scaler",  StandardScaler()),
    ("svc_clf", SVC(kernel="sigmoid", C=1, coef0=1, degree=3, random_state=42, class_weight="balanced")),
])
svc.fit(X_train_svc, y_train_svc)

val_acc  = metrics.accuracy_score(y_val_svc,  svc.predict(X_val_svc))
test_acc = metrics.accuracy_score(y_test_svc, svc.predict(X_test_svc))

print(f"SVC — val accuracy: {val_acc:.4f}  |  test accuracy: {test_acc:.4f}")

In [ ]:
print(metrics.classification_report(y_test_svc, svc.predict(X_test_svc), zero_division=0))

## 4. Naive Bayes

Multinomial Naive Bayes requires non-negative features. Note that `train_test_split` does not accept a `class_weight` parameter — class weighting is handled at the model level for applicable classifiers.

In [ ]:
X_train_nb, X_val_nb, X_test_nb, y_train_nb, y_val_nb, y_test_nb = split(
    df, TARGET, random_state=42
)

nb = make_pipeline(MultinomialNB())
nb.fit(X_train_nb, y_train_nb)

test_acc = metrics.accuracy_score(y_test_nb, nb.predict(X_test_nb))
print(f"Naive Bayes — test accuracy: {test_acc:.4f}")
print("With a score near 0.47, the Naive Bayes classifier performed terribly on this dataset.")

In [ ]:
print(metrics.classification_report(y_test_nb, nb.predict(X_test_nb), zero_division=0))

## 5. Logistic Regression

The logistic model achieves an accuracy score slightly above the baseline. Looking at the scatter plot of training labels below, it is somewhat surprising that the model can beat the baseline at all — many of the data points appear right on top of each other, making it hard to separate death from non-death scenes.

In [ ]:
X_train_lg, X_val_lg, X_test_lg, y_train_lg, y_val_lg, y_test_lg = split(
    df, TARGET, random_state=42
)

lgr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lgr.fit(X_train_lg, y_train_lg)

test_acc = metrics.accuracy_score(y_test_lg, lgr.predict(X_test_lg))
print(f"Logistic Regression — test accuracy: {test_acc:.4f}")

In [ ]:
print(metrics.classification_report(y_test_lg, lgr.predict(X_test_lg), zero_division=0))

In [ ]:
plt.scatter(X_train_lg.index.values, y_train_lg, alpha=0.3, s=5)
plt.xlabel('Sample Index')
plt.ylabel('death_in_scene')
plt.title('Training Labels — Scene Death\n(heavy overlap reflects class imbalance)')
plt.tight_layout()
plt.show()

## Results Summary

In [ ]:
results = {
    "Baseline (majority class)": scenes_baseline,
    "Decision Tree":             metrics.accuracy_score(y_test,     dt.predict(X_test)),
    "Random Forest":             metrics.accuracy_score(y_test_rf,  forest.predict(X_test_rf)),
    "SVC":                       metrics.accuracy_score(y_test_svc, svc.predict(X_test_svc)),
    "Naive Bayes":               metrics.accuracy_score(y_test_nb,  nb.predict(X_test_nb)),
    "Logistic Regression":       metrics.accuracy_score(y_test_lg,  lgr.predict(X_test_lg)),
}

results_df = pd.DataFrame.from_dict(results, orient='index', columns=['Test Accuracy'])
results_df = results_df.sort_values('Test Accuracy', ascending=False)

display(results_df)

results_df['Test Accuracy'].plot.barh()
plt.xlabel('Accuracy')
plt.axvline(scenes_baseline, color='grey', linestyle='--', label='Baseline')
plt.title('Scene Death Prediction — Model Comparison')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Episode-Level Evaluation

The best scene-level model (Random Forest) is used to predict `death_in_scene` for every scene in the dataset. Predictions are then aggregated by episode and compared against the ground-truth deaths-per-episode data.

This was the original motivation for the scene-level model: instead of predicting episode deaths directly (which the episode DataFrame does not have enough features to support well), I predict at the scene level and roll the results up.

In [ ]:
# Prepare full feature matrix (same preprocessing as training)
X_all = df.drop(columns=[TARGET]).select_dtypes(include=["number", "bool"])

scenes_with_preds = scenes.copy()
scenes_with_preds['predicted_death'] = forest.predict(X_all)

In [ ]:
# Aggregate predicted deaths by episode
predicted_per_episode = (
    scenes_with_preds
    .groupby(['seasonNum', 'episodeNum'])['predicted_death']
    .sum()
    .reset_index()
    .rename(columns={'predicted_death': 'predictedDeaths'})
)

# Aggregate actual deaths by episode
actual_per_episode = (
    scenes
    .groupby(['seasonNum', 'episodeNum'])['death_in_scene']
    .sum()
    .reset_index()
    .rename(columns={'death_in_scene': 'actualDeaths'})
)

episode_comparison = actual_per_episode.merge(predicted_per_episode, on=['seasonNum', 'episodeNum'])
episode_comparison.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

idx = range(len(episode_comparison))
ax.scatter(idx, episode_comparison['actualDeaths'],    s=15, c='b', marker='s', label='Actual Deaths per Episode')
ax.scatter(idx, episode_comparison['predictedDeaths'], s=15, c='r', marker='o', label='Predicted Deaths per Episode')
ax.legend(loc='upper left')

ax.set_xlabel('Episode Number')
ax.set_ylabel('Number of Deaths')
ax.set_title('Actual vs. Predicted Deaths per Episode in Game of Thrones (Random Forest)')
plt.tight_layout()
plt.show()

In [ ]:
total_actual    = episode_comparison['actualDeaths'].sum()
total_predicted = episode_comparison['predictedDeaths'].sum()
total_error     = (episode_comparison['actualDeaths'] - episode_comparison['predictedDeaths']).sum()
error_fraction  = total_error / total_actual

print(f"Average deaths per episode (actual):    {episode_comparison['actualDeaths'].mean():.2f}")
print(f"Total deaths (actual):                  {total_actual}")
print(f"Total deaths (predicted):               {total_predicted}")
print(f"Total error (actual - predicted):       {total_error}")
print(f"Error as fraction of total deaths:      {error_fraction:.4f}")

After balancing the classes, the results were noticeably better, though the model still tends to underestimate deaths per episode. This is not too surprising: since the classes are so imbalanced, even small numbers of false negatives (scenes where a death occurred but the model predicts no death) have a large effect at the episode level — especially because most episodes contain fewer than 4 deaths, so even one wrong prediction matters.

## Conclusion

- The **Random Forest** was the best scene death predictor, achieving a test accuracy of ~0.965 — slightly above the 0.952 majority-class baseline.
- The **SVC** (sigmoid kernel) also beat the baseline at ~0.952.
- The **Decision Tree** and **Logistic Regression** performed near or slightly below the baseline in standard accuracy, though with `class_weight='balanced'` they show better recall on the minority class.
- **Naive Bayes** performed very poorly (~0.47) — far below the baseline.

The class imbalance (fewer than 5% of scenes contain a death) is the dominant challenge. Even the best model underestimates episode death counts, because each missed death prediction carries significant weight at the aggregate level.

Future directions:
- Apply SMOTE or other oversampling strategies to the training data.
- Tune the classification threshold rather than defaulting to 0.5.
- Perform PCA to reduce the feature space and identify the most discriminative features.